# MeatGuard II — 02 · Distillation, compression and export

1. **Distillation** from the teacher selected in notebook 00 (`runs/teacher/teacher_selection.json`):
   loss = α·CE + (1 − α)·T²·KL + β·feature alignment (α = 0.3, T = 4, β = 0.1), best epoch chosen on validation macro-F1.
2. **Compression**: structured L1 channel pruning (30%, dependency-aware, transformer block kept) and fine-tuning.
3. **Export**: ONNX and a full-integer INT8 TFLite model, calibrated on training images; its agreement with the
   PyTorch model is checked on validation images.

**Saved:** `runs/student/distilled/` (`best.pt`, `history.csv`, `summary.json`), `runs/student/pruned/`
(`best_model.pt`, `history.csv`, `summary.json` with the cost before/after pruning and the channel counts) and
`runs/student/export/` (`meatguard_s.onnx`, `meatguard_s_int8.tflite`, `summary.json` with file sizes and INT8 vs FP32
validation metrics). The `.tflite` file is the model for the Android app.

## 1. Setup
Upload the whole `MeatGuard_II` folder to Google Drive and the data zip (`crops/`, `masks/`, `sessions.csv`) next to
it, then edit the two paths below if yours differ. Results are written to `MeatGuard_II/runs/` on Drive, so they
survive disconnects; re-running a training cell resumes from the last saved epoch.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- paths: edit these -------------------------------------------------------------------------
REPO_DIR = "/content/drive/MyDrive/MeatGuard_II" if IN_COLAB else os.path.abspath("..")   # the codebase on Drive
DATA_SRC = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "data/processed/full"  # zip or folder: crops/, masks/, sessions.csv
# -------------------------------------------------------------------------------------------------
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC     # local copy on the Colab VM (Drive is slow for many small files)
RUNS = os.path.join(REPO_DIR, "runs")
WORKERS = 4 if IN_COLAB else 0
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if IN_COLAB:
    !pip -q install torch-pruning shap onnx onnxruntime onnxscript onnx2tf onnx_graphsurgeon sng4onnx ai_edge_litert psutil
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data

In [ ]:
# Copy the data to the VM once, then check it: every session in one partition, every listed image present
import zipfile, shutil
import pandas as pd
from pathlib import Path
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(DATA_ROOT)
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
if not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):      # zip made from the parent folder
    nested = [p.parent for p in Path(DATA_ROOT).glob("*/sessions.csv")]
    DATA_ROOT = str(nested[0]) if nested else DATA_ROOT
CROPS, SPLIT, MASKS = (os.path.join(DATA_ROOT, d) for d in ("crops", "sessions.csv", "masks"))

from src.train_teacher import check_split
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", check_split(SPLIT), "| masks/ present:", os.path.isdir(MASKS))
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"

## 3. Knowledge distillation
Smoke test first (one minute), then the full run. Re-run the second cell to resume after a disconnect.

In [ ]:
TEACHER_SEL = os.path.join(RUNS, "teacher", "teacher_selection.json")
assert os.path.exists(TEACHER_SEL), "Run notebook 00 first"
!python -m src.train_student distill --data {CROPS} --split-file {SPLIT} --teacher {TEACHER_SEL} \
    --out /content/smoke_student --epochs 1 --num-workers {WORKERS} --smoke --restart

In [ ]:
!python -m src.train_student distill --data {CROPS} --split-file {SPLIT} --teacher {TEACHER_SEL} \
    --out {RUNS}/student --num-workers {WORKERS}

In [ ]:
import matplotlib.pyplot as plt
def plot_curves(run_dirs, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for d in run_dirs:
        h = Path(d) / "history.csv"
        if not h.exists():
            continue
        df = pd.read_csv(h)
        for ax, col in zip(axes, ["train_loss", "val_macro_f1", "val_recall_spoiled"]):
            ax.plot(df["epoch"] + 1, df[col], label=Path(d).name)
    for ax, t in zip(axes, ["training loss", "validation macro-F1", "validation spoiled recall"]):
        ax.set_title(f"{title} {t}".strip()); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    axes[1].legend(); plt.tight_layout(); plt.show()

plot_curves([os.path.join(RUNS, "student", d) for d in ("distilled", "scratch")], "student")

## 4. Structured pruning and fine-tuning

In [ ]:
!python -m src.train_student compress --data {CROPS} --split-file {SPLIT} \
    --student {RUNS}/student/distilled/best.pt --out {RUNS}/student/pruned --num-workers {WORKERS}
import json
s = json.load(open(os.path.join(RUNS, "student", "pruned", "summary.json")))
print("cost before:", s["cost_before"], "| after:", s["cost_after"])
print("val macro-F1: distilled", round(s["val_before_pruning"]["macro_f1"], 4),
      "| pruned, before fine-tuning", round(s["val_after_pruning_before_finetune"]["macro_f1"], 4),
      "| after fine-tuning", round(s["val"]["macro_f1"], 4))

## 5. Export: ONNX and INT8 TFLite

In [ ]:
!python -m src.train_student export --data {CROPS} --split-file {SPLIT} \
    --model {RUNS}/student/pruned/best_model.pt --out {RUNS}/student/export --num-workers {WORKERS}
e = json.load(open(os.path.join(RUNS, "student", "export", "summary.json")))
print("sizes (MB):", e["size_mb"], "| INT8/FP32 agreement on validation:", round(e["val_prediction_agreement"], 4))

In [ ]:
# Bundle all result files (JSON/CSV, no weights) into one zip to share: runs/results_bundle.zip on Drive
!python -m src.collect_results --runs {RUNS} --out {RUNS}/results_bundle.zip